# Description

- Section 1: Process Input Hurricane Raw Data (in the format of one Hurricane Per Sheet); treats missing values; and process the Announcement Time into standardized format accordingly
- Section 2: Scrap all Public Adivisories Time Information from the provided website and process them into standardized format accordingly

In [ ]:
import pandas as pd
import numpy as np
import re
from urllib.request import urlopen
import json
import datetime as dt
from bs4 import BeautifulSoup


## Set up pandas df parameters
# To display all rows:
pd.set_option('display.max_rows', None)

# To display all columns:
pd.set_option('display.max_columns', None)

# run this notebook from the pipeline/ folder
folder_path = "."

# Section 1: Process Hurricane Input data

## 1.1 Create a combined file and process county information

In [ ]:
# declare input file path variable
yourFilePath = f"{folder_path}/data/HEvOD_2014-2026_Website.xlsx"


rawData = pd.ExcelFile(yourFilePath)
result_df = pd.DataFrame() # empty dataframe as the base


# Separators: ", and " / " and " / "&"  ->  ","   (keeps "King and Queen" intact)
RE_SEPARATOR = r',\s*and\s+|(?<!King)\s+and\s+(?!Queen)|\s*&\s*'

# Suffix words removed (plural forms first); "James City" is kept intact
RE_SUFFIX = r'(?i)\s+(?:counties|county|parishes|parish)\b|(?<!James)\s+city\b'

for sheet_name in rawData.sheet_names:
    if ("pivot" not in sheet_name.lower()) and (sheet_name != 'Read Me'):  # Only want to process sheets that are not pivot sheets
        print(f"Processing sheet {sheet_name}")

        df_county_detail = pd.read_excel(yourFilePath, sheet_name=sheet_name)

        if len(df_county_detail) > 0:
            # drop null values if any in Order Type
            df_county_detail.dropna(subset=['Order Type'], inplace=True)

            df_county_detail = df_county_detail.applymap(lambda x: x.strip() if isinstance(x, str) else x)

            # make the event name unique across years: "Hurricane Isaias" -> "Hurricane Isaias (2020)"
            sheet_year = sheet_name.split()[0]
            df_county_detail['Event Name'] = df_county_detail['Event Name'].astype(str).str.strip() + f" ({sheet_year})"
            
            #----------------------------------------------------------------------------------------------------------
            # remove the words/connectors from County
            df_county_detail['County'] = (df_county_detail['County'].str.replace(RE_SEPARATOR, ',', regex=True).str.replace(RE_SUFFIX, '', regex=True))

            # missing county -> "UNKNOWN"
            df_county_detail['County'] = df_county_detail['County'].fillna('UNKNOWN')

            # one row per county
            df_county_detail['County'] = df_county_detail['County'].str.split(',').map(
                lambda items: [i.strip() for i in items if i.strip()] or ['UNKNOWN'])
            df_county_detail_exploded = df_county_detail.explode('County')

            result_df = pd.concat([result_df, df_county_detail_exploded])
            
# Select only columns we want
result_df = result_df[['Event Name', 'Announcement Date', 'Announcement Time', 'Time Zone', 'State', 'County',
                                      'Order Type', 'Order Type Code', 'Evacuation Area', 'Effective Date', 'Effective Time']]

# rename column "State" to "States" and "County" to "Counties Short"
result_df.rename(columns={"State": "States", "County": "Counties Short"}, inplace=True)

# close the pd.ExcelFile (which works like a document open as opposed to pd.read_excel())
rawData.close()

## 1.2 Handle missing information

- Missing values in `Time Zone` are assigned a zone based on County and State. Use time zone table to fill missing content. Reference links: https://www.weather.gov/gis/Counties; https://www.law.cornell.edu/cfr/text/49/71.5; https://lists.iana.org/hyperkitty/list/tz@iana.org/thread/FAITOSL7K57L3UZOFCDSGNT4GZJKC5ZN/; https://github.com/geanders/countytimezones
- Handle `Announcement Time` and `Effective Time` by treating any time format to actual time and missing values or invalid strings to 00:00 AM.
- Handle `Announcement Date` and `Effective Date`

In [ ]:
# TIME ZONE ==================================================

TZ_TABLES = fr"{folder_path}/data/time_zones_by_region/timezone_lookup_tables.xlsx"

# abbreviations -> named zones
TZ_ABBR_MAP = {
    'EST': 'Eastern Time',  'EDT': 'Eastern Time',
    'CST': 'Central Time',  'CDT': 'Central Time',
    'MST': 'Mountain Time', 'MDT': 'Mountain Time',
    'PST': 'Pacific Time',  'PDT': 'Pacific Time',
    'AKST': 'Alaska Time',  'AKDT': 'Alaska Time',
    'HST': 'Hawaii-Aleutian Time', 'HDT': 'Hawaii-Aleutian Time',
    'HAST': 'Hawaii-Aleutian Time', 'HADT': 'Hawaii-Aleutian Time',
}
NAMED_ZONES = set(TZ_ABBR_MAP.values())

def county_key(s):
    """Same key used in the lookup tables: lowercase letters/digits, no county/parish/etc. words."""
    if pd.isna(s):
        return ''
    s = str(s).lower().replace('saint', 'st').replace('ste.', 'st')
    s = re.sub(r'\b(county|parish|borough|census area|municipality|city and borough|city)\b', '', s)
    return re.sub(r'[^a-z0-9]', '', s)


# lookup tables
state_tbl  = pd.read_excel(TZ_TABLES, sheet_name='State_Default')
state_map  = state_tbl.set_index('State')['Default Time Zone'].to_dict()

county_tbl = pd.read_excel(TZ_TABLES, sheet_name='County_Overrides', dtype={'County FIPS': str})
county_map = county_tbl.set_index(['State', 'County Key'])['Time Zone'].to_dict()


# standardise existing values
tz_clean = result_df['Time Zone'].astype('string').str.strip().str.upper()
mapped = tz_clean.map(TZ_ABBR_MAP)
mapped = mapped.fillna(result_df['Time Zone'].where(result_df['Time Zone'].isin(NAMED_ZONES)))   # already a named zone

# 4) fill missing: county override first, then state default
keys = list(zip(result_df['States'].str.strip().str.upper(), result_df['Counties Short'].map(county_key)))
by_county = pd.Series([county_map.get(k) for k in keys], index=result_df.index)
by_state  = result_df['States'].str.strip().str.upper().map(state_map)

result_df['Time Zone'] = mapped.fillna(by_county).fillna(by_state)

print("Time Zone still missing:", result_df['Time Zone'].isna().sum())

In [ ]:
# ANNOUNCEMENT TIME & EFFECTIVE TIME ==================================================

'''
Before assigning 00:00 to missing values / junk entries, fill `Effective Time` with `Announcement Time` and vice versa if any one of it is not missing / junk.
'''

TIME_FORMATS = ['%H:%M:%S', '%H:%M', '%I:%M:%S %p', '%I:%M %p', '%I %p']
MIDNIGHT = dt.time(0, 0)

def parse_time(x):
    """Return a datetime.time for a valid time, otherwise None (missing / junk)."""
    if isinstance(x, (dt.datetime, pd.Timestamp)):
        return None if pd.isna(x) else x.time()
    if isinstance(x, dt.time):
        return x
    if isinstance(x, str):
        s = x.strip().upper().replace('.', '')
        s = s.replace('AM', ' AM').replace('PM', ' PM')
        s = ' '.join(s.split())
        for fmt in TIME_FORMATS:
            try:
                return dt.datetime.strptime(s, fmt).time()
            except ValueError:
                pass
    return None

ann = result_df['Announcement Time'].map(parse_time)
eff = result_df['Effective Time'].map(parse_time)

# fill each from the other when only one is valid
ann_filled = ann.where(ann.notna(), eff)
eff_filled = eff.where(eff.notna(), ann)

# anything still missing -> 00:00:00
result_df['Announcement Time'] = ann_filled.map(lambda t: t if pd.notna(t) else MIDNIGHT)
result_df['Effective Time']    = eff_filled.map(lambda t: t if pd.notna(t) else MIDNIGHT)

In [ ]:
# ANNOUNCEMENT DATE & EFFECTIVE DATE ==================================================

# make sure both columns are real datetimes (text/junk dates become NaT)
result_df['Announcement Date'] = pd.to_datetime(result_df['Announcement Date'], errors='coerce')
result_df['Effective Date']    = pd.to_datetime(result_df['Effective Date'],    errors='coerce')

# drop rows with no Announcement Date
n_before = len(result_df)
result_df = result_df.dropna(subset=['Announcement Date']).reset_index(drop=True)
print(f"Dropped {n_before - len(result_df)} rows with missing Announcement Date")

# missing Effective Date -> Announcement Date
result_df['Effective Date'] = result_df['Effective Date'].fillna(result_df['Announcement Date'])

In [ ]:
# Add ORDER ANNOUNCEMENT TIME ==================================================

ZONE_TO_IANA = {
    'Eastern Time': 'America/New_York',
    'Central Time': 'America/Chicago',
    'Mountain Time': 'America/Denver',
    'Pacific Time': 'America/Los_Angeles',
    'Alaska Time': 'America/Anchorage',
    'Hawaii-Aleutian Time': 'Pacific/Honolulu',
}
iana = result_df['Time Zone'].map(ZONE_TO_IANA)

# Arizona does not observe DST (except the Navajo Nation counties)
az = (result_df['States'] == 'AZ') & ~result_df['Counties Short'].map(county_key).isin(['apache', 'coconino', 'navajo'])
iana = iana.mask(az, 'America/Phoenix')

# time exactly as announced (no time zone attached)
result_df['Order Announcement Time'] = pd.to_datetime(
    result_df['Announcement Date'].dt.strftime('%Y-%m-%d') + ' ' + result_df['Announcement Time'].astype(str),
    format='%Y-%m-%d %H:%M:%S')

# time in UTC, so events in different zones can be compared / sorted
pieces = []
for zone, local in result_df['Order Announcement Time'].groupby(iana):
    pieces.append(local.dt.tz_localize(zone, ambiguous=True, nonexistent='shift_forward').dt.tz_convert('UTC'))
result_df['Order Announcement Time UTC'] = (pd.concat(pieces).reindex(result_df.index) if pieces else pd.NaT)

In [ ]:
result_dedup = result_df.drop_duplicates().copy()
result_dedup['Year'] = result_dedup['Announcement Date'].dt.year.astype(int)

In [ ]:
# keep the original text, then regroup
result_dedup['Order Type Raw'] = result_dedup['Order Type']

ORDER_TYPE_GROUPS = {
    # --- evacuation orders ---
    "Mandatory evacuation order":              "Mandatory evacuation order",
    "Voluntary evacuation order":              "Voluntary evacuation order",

    # --- evacuation orders lifted ---
    "Mandatory evacuation order lifted":       "Evacuation order lifted",
    "Mandatory evacuation order rescinded":    "Evacuation order lifted",   # rescinded = lifted
    "Voluntary evacuation order lifted":       "Evacuation order lifted",
    "Evacuation order lifted":                 "Evacuation order lifted",

    # --- state of emergency: declared / updated / expanded / extended; federal & local emergency -> one group ---
    "State of emergency":                      "Emergency declared",
    "State of emergency (updated)":            "Emergency declared",
    "State of emergency updated":              "Emergency declared",
    "State of emergency expanded":             "Emergency declared",
    "State of emergency extended":             "Emergency declared",
    "State of emergency extended for 60 days": "Emergency declared",
    "Final Extension of State of Emergency":   "Emergency declared",
    "State of local emergency":                "Emergency declared",
    "Local state of emergency":                "Emergency declared",
    "Federal State of Emergency":              "Emergency declared",

    # --- keep separate if order is different in purpose or has different issuer ---
    "State of emergency lifted":               "State of emergency lifted",
    "Notice":                                  "Notice",
}

unmapped = set(result_dedup['Order Type Raw'].dropna().unique()) - set(ORDER_TYPE_GROUPS)
assert not unmapped, f"Add these to ORDER_TYPE_GROUPS: {unmapped}"

result_dedup['Order Type'] = result_dedup['Order Type Raw'].map(ORDER_TYPE_GROUPS)

## 1.3 Merge county codes

In [ ]:
# load the merging dataset
with open(fr"{folder_path}/geojson-counties-fips.json") as response:
    counties = json.load(response)

# parse the geojson file into a pd.DataFrame
feature = counties['features']

state_name_col = [info['properties']['STATE'] for info in feature]
region_name_col = [info['properties']['NAME'] for info in feature]
LSAD_type_col = [info['properties']['LSAD'] for info in feature]
FIPS_type_col = [info['id'] for info in feature]

region_info = pd.DataFrame({
    'States_FIPS': state_name_col,
    'Counties Short': region_name_col,
    'region_type(LSAD)': LSAD_type_col,
    'FIPS': FIPS_type_col
})


# state FIPS code -> state name / abbreviation (columns: States_FIPS, stname, States)
state_name_to_fips = pd.read_csv(f"{folder_path}/state_fips.csv")

# convert the 'region_info' dataframe's state_name column to integer
region_info['States_FIPS'] = region_info['States_FIPS'].astype(np.int64)

# merge the two datasets together
final_region_ds = region_info.merge(state_name_to_fips, how="left", on='States_FIPS')

# strip of extra space for the region and state column
final_region_ds['States'] = final_region_ds['States'].str.strip()
final_region_ds['Counties Short'] = final_region_ds['Counties Short'].str.strip()

final_region_ds['States'] = final_region_ds['States'].str.strip()

In [ ]:
# merge dataset with regional code with the processed hurricane dataset
final_merged = result_dedup.merge(final_region_ds, how="left", on=['States', 'Counties Short'], indicator=True)

# state-level info for every state, to fill rows with no county match (e.g. UNKNOWN)
state_lookup = final_region_ds.drop_duplicates('States').set_index('States')[['States_FIPS', 'stname']]
for col in ['States_FIPS', 'stname']:
    final_merged[col] = final_merged[col].fillna(final_merged['States'].map(state_lookup[col]))
final_merged['States_FIPS'] = final_merged['States_FIPS'].astype('Int64')

final_merged['Evacuation Area'].fillna("Not Available", inplace = True)

final_merged = final_merged[['Year','Event Name','Order Announcement Time','Order Announcement Time UTC', 
                            'Time Zone','States','Counties Short','Order Type','Order Type Code',
                            'Evacuation Area','Effective Date','Effective Time','States_FIPS','FIPS']]


# order events chronologically (by their first announcement), then rows within each event by announcement time UTC
event_start = final_merged.groupby(['Year', 'Event Name'])['Order Announcement Time UTC'].transform('min')

final_merged = (final_merged
                .assign(_event_start=event_start)
                .sort_values(['_event_start', 'Event Name', 'Order Announcement Time UTC'], kind='stable')
                .drop(columns='_event_start').reset_index(drop=True))

In [ ]:
final_merged.to_parquet(f'{folder_path}/data/temp/hurricane_processed.parquet', index=False)

# out = final_merged.copy()
# out['Order Announcement Time UTC'] = out['Order Announcement Time UTC'].dt.tz_localize(None)
# out.to_excel(f'{folder_path}/data/temp/hurricane_processed.xlsx', index=False)

# Section 2: Scrap public advisory date & time

In [ ]:
rawData = pd.ExcelFile(yourFilePath) # 'yourFilePath' variable is declared above, which is your main Hurricane data file path

event_list = []
year_list = []

for sheet_name in rawData.sheet_names:
    if ("pivot" not in sheet_name.lower()) and (sheet_name != 'Read Me'):  # Only want to process sheets that are not pivot sheets
        year_list.append(sheet_name.split()[0])
        event_list.append(sheet_name.split()[1])

In [ ]:
# empty DataFrame
PAD = pd.DataFrame()

# link text looks like "1:\xa01100 AM EDT" (2021 and earlier, local time) or "1:\xa02100 UTC" (2022 onward)
RE_ADVISORY = re.compile(r':\s*(\d{3,4})\s*(AM|PM)?\s*([A-Z]{3,4})\s*$')

for i in range(len(event_list)):
    event = event_list[i].upper()
    print(f"Processing {event}")
    
    event_full = "Hurricane " + event.lower().capitalize() + f" ({year_list[i]})"
    url = f"https://www.nhc.noaa.gov/archive/{year_list[i]}/{event}.shtml?"
    page = urlopen(url)
    html = page.read().decode("utf-8")
    soup = BeautifulSoup(html, "html.parser")
    
    PAD_df = dict()

    PAD_df['Event Name'] = []
    PAD_df['Public Advisories Date'] = []
    PAD_df['Public Advisories Time'] = []
    PAD_df['Public Advisories Timezone'] = []

    current_date = None

    for el in soup.find_all(["td", "a"]):
        if el.name == "td":
            if el.get('colspan') == '4' and el.i is not None:
                raw_date = " ".join(el.i.get_text().replace(",", " ").split())
                current_date = dt.datetime.strptime(raw_date, "%A %B %d %Y").strftime("%m/%d/%Y")
            continue

        href = el.get('href', '')
        if current_date is None or '.public' not in href:   # also excludes ".update" links
            continue

        m = RE_ADVISORY.search(el.get_text().replace("\xa0", " ").strip())
        if m is None:
            continue
        digits, meridiem, time_zone = m.groups()
        digits = digits.zfill(4)

        # store as 24-hour "HH:MM"
        if meridiem:
            parsed = dt.datetime.strptime(f"{digits}{meridiem}", "%I%M%p")
        else:
            parsed = dt.datetime.strptime(digits, "%H%M")

        PAD_df['Event Name'].append(event_full)
        PAD_df['Public Advisories Date'].append(current_date)
        PAD_df['Public Advisories Time'].append(parsed.strftime("%H:%M"))
        PAD_df['Public Advisories Timezone'].append(time_zone)

    PAD_df = pd.DataFrame(PAD_df)
    
    PAD = pd.concat([PAD, PAD_df], ignore_index=True)


In [ ]:
# ---- match hurricane_processed.parquet formatting ----
ABBR_TO_ZONE = {
    'EST': 'Eastern Time',  'EDT': 'Eastern Time',
    'CST': 'Central Time',  'CDT': 'Central Time',
    'MST': 'Mountain Time', 'MDT': 'Mountain Time',
    'PST': 'Pacific Time',  'PDT': 'Pacific Time',
    'AKST': 'Alaska Time',  'AKDT': 'Alaska Time',
    'HST': 'Hawaii-Aleutian Time', 'HDT': 'Hawaii-Aleutian Time',
    'AST': 'Atlantic Time', 'ADT': 'Atlantic Time',     # Puerto Rico / USVI (not in the six named zones)
    'UTC': 'UTC',                                       # NHC lists 2022+ public advisories in UTC
}
# UTC offset (hours) implied by each abbreviation
ABBR_OFFSET = {'EST': -5, 'EDT': -4, 'CST': -6, 'CDT': -5, 'MST': -7, 'MDT': -6,
               'PST': -8, 'PDT': -7, 'AKST': -9, 'AKDT': -8, 'HST': -10, 'HDT': -9,
               'AST': -4, 'ADT': -3, 'UTC': 0}

abbr = PAD['Public Advisories Timezone'].str.strip().str.upper()

# clock time as listed by NHC (local for older events, UTC for 2022+), no time zone attached
# ('Public Advisories Time' is 24-hour "HH:MM", produced by the scraping cell)
PAD['Public Advisories Time'] = pd.to_datetime(
    PAD['Public Advisories Date'] + ' ' + PAD['Public Advisories Time'],
    format='%m/%d/%Y %H:%M', errors='coerce')

# the same moment in UTC, timezone-aware (like 'Order Announcement Time UTC')
PAD['Public Advisories Time UTC'] = (
    PAD['Public Advisories Time'] - pd.to_timedelta(abbr.map(ABBR_OFFSET), unit='h')
).dt.tz_localize('UTC')

# named zone (like 'Time Zone')
PAD['Time Zone'] = abbr.map(ABBR_TO_ZONE)

PAD['Year'] = PAD['Public Advisories Time'].dt.year.astype('int32')

print("Unparsed times:", PAD['Public Advisories Time'].isna().sum(),
      "| Unrecognised zones:", sorted(abbr[PAD['Time Zone'].isna()].unique()))

PAD = PAD[['Year', 'Event Name', 'Public Advisories Time', 'Public Advisories Time UTC', 'Time Zone']]

In [ ]:
PAD.to_parquet(f'{folder_path}/data/temp/PAD_processed.parquet', index=False)

# PAD_out = PAD.copy()
# PAD_out['Public Advisories Time UTC'] = PAD_out['Public Advisories Time UTC'].dt.tz_localize(None)
# PAD_out.to_excel(f'{folder_path}/data/temp/PAD_processed.xlsx', index=False)